# Arboili Data Collection

This notebook downloads and validates all data sources for the project.

| Section | Data source | Module |
|---|---|---|
| 1 | Epidemiological data (SINAN + SIVEP) | `src/epidemiological` |
| 2 | Google Trends — search index | `src/google_trends` |
| 3 | Google Trends — related topics & queries | `src/google_trends` |
| 4 | Climate data (ERA5-Land) | `src/collection/climate` |
| 5 | Health bulletins (Boletins Epidemiológicos) | `src/bulletins` |
| 6 | EBC news articles (Agência Brasil) | `src/ebc` |
| 7 | Validation | all modules |

Run the cells in order. Each section is independent — skip any section whose data is already downloaded.

**Settings come from [`config.yml`](config.yml)** — the same file the `arboili` CLI reads, so the notebook and the command line always agree. To change a year range, a request delay, or an output directory, edit `config.yml` rather than these cells. To override a value for one run only, pass it directly to `extract()`: the explicit argument wins.

Each section below shows its CLI equivalent, if you'd rather run it from a terminal:

```bash
uv run arboili --list      # show what config.yml currently specifies
uv run arboili all         # every section except 3 (the ~18 h job)
```


---
## 0 — Setup

In [ ]:
import sys
from pathlib import Path

# Make src/ importable as a package root
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config

# Every extraction cell below reads its settings from here
CFG = load_config()
DATA_DIR = CFG.data_dir

DIRS = {
    "epidemiological":       DATA_DIR / "epidemiological",
    "epidemiological/SINAN": DATA_DIR / "epidemiological" / "SINAN",
    "epidemiological/SIVEP": DATA_DIR / "epidemiological" / "SIVEP",
    "google_trends":         DATA_DIR / "google_trends",
    "climate":               DATA_DIR / "climate",
    "bulletins":             DATA_DIR / "bulletins",
    "news":                  DATA_DIR / "news",
}
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Config file: ", CFG.path)
print("Data dir:    ", DATA_DIR)
print("\nConfigured sources:")
for source in ["sinan", "gt_search", "gt_related", "era5", "inmet", "bulletins", "ebc"]:
    print(f"  {source:<11} {CFG.source(source)}")


---
## 1 — Epidemiological Data (SINAN Dengue)

Downloads yearly dengue case CSVs from the Brazilian Ministry of Health S3 bucket via `src/epidemiological/sinan_dengue.py`. The year range comes from `sources.sinan` in `config.yml`.

Already-downloaded files are skipped automatically. A `manifest.csv` and `failures.csv` are written after every run so you can track what was retrieved. To retry only failed years, re-run the cell — existing years will not be re-fetched.

**CLI equivalent:** `uv run arboili sinan`

> **Note:** Aggregation of the raw per-row CSVs into the final `SINAN_dengue_cases.csv.gz` is still done by the R script `r/scripts/2_transform_sinan_data.R`. Run it after this cell if you need the aggregated file.

In [ ]:
from src.collection.epidemiological.sinan_dengue import extract

cfg = CFG.source("sinan")

result = extract(
    out_dir=CFG.resolve(cfg["out_dir"]),
    from_year=cfg["from_year"],
    to_year=cfg["to_year"],
    keep_zip=cfg.get("keep_zip", False),
)
print(result)


---
## 2 — Google Trends: Search Index

Downloads a 5-year rolling window of weekly search-interest values for each disease/symptom group × Brazilian state combination using pytrends (no API key required). The window end date and request delay come from `sources.gt_search` in `config.yml`.

**Output:** `data/google_trends/GoogleTrends_search.csv`

The run takes a few minutes for the full dataset. If a 429 (rate-limit) error occurs, the script saves progress and stops — re-run the cell to continue from where it left off, or raise `sleep` in `config.yml` first.

**CLI equivalent:** `uv run arboili gt-search`

In [ ]:
from src.config import as_date
from src.collection.google_trends.extract_gt_search import extract

cfg = CFG.source("gt_search")

result = extract(
    # Fixed reference date (from config.yml) so the 5-year window is reproducible
    reference_date=as_date(cfg["reference_date"]),
    out_dir=CFG.resolve(cfg["out_dir"]),
    popular_terms_path=CFG.reference("popular_terms"),
    fu_path=CFG.reference("federative_units"),
    sleep=cfg["sleep"],
)
print(result)


---
## 3 — Google Trends: Related Topics & Queries

Downloads monthly related topics and queries for the 4 disease groups × 27 UFs + BR, from the `start_month` set in `sources.gt_related` in `config.yml`.

**Outputs:**
- `data/google_trends/GoogleTrends_related_topic.csv`
- `data/google_trends/GoogleTrends_related_query.csv`

> **Runtime warning:** ~13 400 requests at 5 s each ≈ 18 hours. The script saves progress after every successful request and on every 429, and can be resumed safely by re-running the cell. Because of its runtime this section is excluded from `uv run arboili all` — start it on its own.

**CLI equivalent:** `uv run arboili gt-related`

In [ ]:
from src.collection.google_trends.extract_gt_related import extract

cfg = CFG.source("gt_related")

result = extract(
    start_month=cfg["start_month"],
    out_dir=CFG.resolve(cfg["out_dir"]),
    fu_path=CFG.reference("federative_units"),
    popular_terms_path=CFG.reference("popular_terms"),
    sleep=cfg["sleep"],
)
print(result)


---
## 4 — Climate Data (ERA5 — Zenodo → ARCO → CDS)

Fastest path first (see `sources.era5.backends` in `config.yml`):

1. **Zenodo** — ERA5-Land municipality daily parquets (HTTP bulk, no key). Covers 1950–2022 + 2025.
2. **ARCO** — Google public ERA5 Zarr for gap years (2023–2024, …). No key, no CDS queue.
3. **CDS** — optional last resort (`backends: [zenodo, arco, cds]`).

**Output:** `data/climate/era5/zenodo/…`, `…/arco/…`, combined `manifest.csv`

> Historical Zenodo files are ~3 GB each (six variables ≈ 18 GB). Re-runs skip files already on disk. Optional INMET: `uv run arboili inmet`.

**CLI equivalent:** `uv run arboili era5`

In [ ]:
import logging
from datetime import datetime
from src.collection.climate.download_era5_data import extract

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")

cfg = CFG.source("era5")

result = extract(
    out_dir=CFG.resolve(cfg["out_dir"]),
    from_year=cfg["from_year"],
    # to_year: null in config.yml means "up to the current year"
    to_year=cfg["to_year"] or datetime.now().year,
    backends=cfg.get("backends", ["zenodo", "arco"]),
    variables=cfg.get("variables"),
    area=cfg.get("area"),
    grid=cfg.get("grid"),
    dataset=cfg.get("dataset", "reanalysis-era5-land"),
    delay=cfg.get("delay", 1.0),
)
print(result)


---
## 5 — Health Bulletins (Boletins Epidemiológicos)

Downloads weekly epidemiological bulletins published by the Brazilian Ministério da Saúde for the year range set in `sources.bulletins` in `config.yml` (`to_year: null` means up to the current year). Already-downloaded PDFs are skipped.

**Output:** `data/bulletins/<year>/<filename>.pdf`

**CLI equivalent:** `uv run arboili bulletins`

In [ ]:
from datetime import datetime
from src.collection.bulletins.download_boletins import extract

cfg = CFG.source("bulletins")

result = extract(
    out_dir=CFG.resolve(cfg["out_dir"]),
    from_year=cfg["from_year"],
    # to_year: null in config.yml means "up to the current year"
    to_year=cfg["to_year"] or datetime.now().year,
)
print(result)


---
## 6 — EBC News Articles (Agência Brasil)

Scrapes news articles from EBC's search engine, one run per term listed under `sources.ebc.queries` in `config.yml`. The scraper is resumable — re-running picks up from the last completed page and skips articles already in the manifest.

**Output:** `data/news/<query>/` (state.json, manifest.jsonl, listings/, articles/)

To add a search term permanently, add it to `sources.ebc.queries`; the cell below and the CLI both pick it up. The second cell is an escape hatch for one-off terms you don't want to record in the config.

**CLI equivalent:** `uv run arboili ebc` (or `uv run arboili ebc --query <term>` for an ad-hoc run)

In [ ]:
import argparse
from src.collection.ebc.scraper import scrape

cfg = CFG.source("ebc")
base_dir = CFG.resolve(cfg["out_dir"])

# One scrape per query listed under sources.ebc.queries in config.yml.
# Each writes to its own subdirectory: data/news/<name>/
for entry in cfg["queries"]:
    name = entry.get("name", entry["query"])
    print(f"\n=== {name} ===")
    result = scrape(argparse.Namespace(
        query=entry["query"],
        output_dir=base_dir / name,
        site=cfg["site"],
        types=cfg["types"],
        per_page=cfg["per_page"],
        max_pages=cfg["max_pages"],
        delay=cfg["delay"],
        restart=False,
        verbose=False,
    ))
    print(result)


In [ ]:
# Ad-hoc scrape of a term that is NOT in config.yml.
# For anything you want to keep, add it to sources.ebc.queries instead —
# the cell above then picks it up, and so does `uv run arboili ebc`.
#
# scrape(argparse.Namespace(
#     query="chikungunya",
#     output_dir=base_dir / "chikungunya",
#     site=cfg["site"],
#     types=cfg["types"],
#     per_page=cfg["per_page"],
#     max_pages=cfg["max_pages"],
#     delay=cfg["delay"],
#     restart=False,
#     verbose=False,
# ))


---
## 7 — Validation

Smoke-tests for each module and a final inventory of all output files.

### 7.1 — Google Trends API helpers

In [ ]:
# Offline tests for gtrends helper functions (no network required)
from src.collection.google_trends.gtrends_api import _to_timeframe, _time_label

cases = [
    ("2020-01", "2024-12"),
    ("2019-06", "2020-06"),
]
for start, end in cases:
    tf = _to_timeframe(start, end)
    label = _time_label(start, end)
    print(f"  {start} → {end}  |  timeframe={tf!r}  label={label!r}")

In [ ]:
# Live smoke test — one small GT request (requires internet)
try:
    from src.collection.google_trends.gtrends_api import try_gtrends_api, _make_client
    client = _make_client()
    df, err = try_gtrends_api(
        topic_keyword="/m/09wsg",  # Dengue topic ID
        geo_location="BR",
        start_date="2024-01",
        end_date="2024-03",
        fun="graph",
        client=client,
        sleep=2,
    )
    if err:
        print(f"Error: {err}")
    else:
        print(f"GT smoke test OK — {len(df)} rows returned")
        print(df.head(3))
except Exception as exc:
    print(f"Skipped (network unavailable): {exc}")

### 7.2 — SINAN dengue URL helpers

In [ ]:
# URL construction test (offline)
from src.collection.epidemiological.sinan_dengue import csv_zip_url

for year in [2010, 2020, 2024]:
    print(f"  {year}  →  {csv_zip_url(year)}")

In [ ]:
# S3 availability probe — checks which years are live (requires internet)
try:
    from src.collection.epidemiological.sinan_dengue import probe_year
    available = [y for y in range(2010, 2027) if probe_year(y)]
    print("Available SINAN dengue years on S3:", available)
except Exception as exc:
    print(f"Skipped (network unavailable): {exc}")

### 7.3 — Output inventory

Final check: list every expected output file with its size and row count.

In [ ]:
import pandas as pd

inventory = {
    "SINAN / manifest":                DATA_DIR / "epidemiological/SINAN/manifest.csv",
    "Google Trends / Search manifest": DATA_DIR / "google_trends/manifest_search.csv",
    "Google Trends / Related manifest":DATA_DIR / "google_trends/manifest_related.csv",
    "Google Trends / Search data":     DATA_DIR / "google_trends/GoogleTrends_search.csv",
    "Google Trends / Related topics":  DATA_DIR / "google_trends/GoogleTrends_related_topic.csv",
    "Google Trends / Related queries": DATA_DIR / "google_trends/GoogleTrends_related_query.csv",
    "ERA5 / manifest":                 DATA_DIR / "climate/era5/manifest.csv",
    "INMET / manifest":                DATA_DIR / "climate/inmet/manifest.csv",
    "Bulletins / manifest":            DATA_DIR / "bulletins/manifest.csv",
}

# One entry per EBC query configured in config.yml
_ebc = CFG.source("ebc")
for entry in _ebc.get("queries", []):
    name = entry.get("name", entry["query"])
    inventory[f"News / {name} manifest"] = CFG.resolve(_ebc["out_dir"]) / name / "manifest.jsonl"

print(f"{'File':<45}  {'Size':>10}  {'Rows':>10}  {'Status summary'}")
print("-" * 90)
for label, path in inventory.items():
    if not path.exists():
        print(f"  {label:<43}  {'MISSING':>10}")
        continue
    size_kb = path.stat().st_size / 1024
    try:
        if path.suffix == ".jsonl":
            rows = sum(1 for _ in open(path, encoding="utf-8"))
            print(f"  {label:<43}  {size_kb:>9.1f}K  {rows:>10,}")
        else:
            df = pd.read_csv(path)
            if "status" in df.columns:
                summary = df["status"].value_counts().to_dict()
                print(f"  {label:<43}  {size_kb:>9.1f}K  {len(df):>10,}  {summary}")
            else:
                print(f"  {label:<43}  {size_kb:>9.1f}K  {len(df):>10,}")
    except Exception as e:
        print(f"  {label:<43}  {size_kb:>9.1f}K  (unreadable: {e})")
